# spinangle — gated spherical nGPT-JEPA vs. official LeWM (Colab GPU)

Official install: `uv` + isolated **Python 3.10** venv + `stable-worldmodel[train]` + the env deps the chosen benchmark uses. **Self-diagnosing**: fragile steps are captured and a per-module import check prints the exact traceback (no hidden errors).

Runtime → GPU. Start with `BENCH='tworoom'` + `EPOCHS=5`.

## ▶️ The big cell (edit config, run)

In [ ]:
#@title 🌀 spinangle: gated spherical nGPT-JEPA vs official LeWM — REAL install + run
import os, subprocess, sys, glob

# ----------------------------- config -----------------------------
BENCH    = "tworoom"   # tworoom (3.4G) | pusht (13G) | reacher (24G) | cube (46G, big disk)
EPOCHS   = 5           # 5 = validate; 100 = matched-compute comparison
VARIANTS = ["official_lewm", "gated_spherical"]
GET_DATA = True
BRANCH   = "claude/upbeat-babbage-kbmgsr"
GH_TOKEN = ""          # only if repo private
# ------------------------------------------------------------------
DATACFG  = {"tworoom": "tworoom", "pusht": "pusht", "reacher": "dmc", "cube": "ogb"}[BENCH]
ENV_DEPS = {"tworoom": "pygame pymunk shapely", "pusht": "pygame pymunk shapely",
            "reacher": "dm_control mujoco", "cube": "ogbench"}[BENCH]
H = "/content/stable-wm"; VENV = "/content/lewmenv"; PY = f"{VENV}/bin/python"
os.environ.update(STABLEWM_HOME=H, MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl")

def run(c, check=True):  # stream (for long steps)
    print(f"\n\033[1;36m$ {c}\033[0m", flush=True)
    return subprocess.run(c, shell=True, check=check).returncode

def cap(c):              # capture + print (so errors are NEVER hidden)
    print(f"\n\033[1;36m$ {c}\033[0m", flush=True)
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print(((r.stdout or "") + (r.stderr or ""))[-7000:]); print("exit", r.returncode)
    return r.returncode

IMPORT_CHECK = (
    f"{PY} - <<'EOF'\n"
    "import importlib, sys, traceback\n"
    "bad = []\n"
    "for m in ['torch','torchvision','hydra','omegaconf','transformers','lightning',"
    "'stable_pretraining','stable_worldmodel']:\n"
    "    try:\n"
    "        x = importlib.import_module(m); print('OK  ', m, getattr(x,'__version__',''))\n"
    "    except Exception:\n"
    "        bad.append(m); print('FAIL', m); traceback.print_exc(file=sys.stdout)\n"
    "import torch; print('CUDA', torch.cuda.is_available()) if 'torch' not in bad else None\n"
    "sys.exit(1 if bad else 0)\n"
    "EOF"
)

try:
    from google.colab import userdata
    GH_TOKEN = GH_TOKEN or (userdata.get("GITHUB_TOKEN") or "")
except Exception:
    pass

run("nvidia-smi -L || echo '⚠️  NO GPU — Runtime > Change runtime type > GPU'", check=False)

# 0) clone ---------------------------------------------------------------------
if not os.path.isdir("/content/spinangle/.git"):
    auth = f"{GH_TOKEN}@" if GH_TOKEN else ""
    run(f"git clone -b {BRANCH} https://{auth}github.com/turtlenottortoise/spinangle.git /content/spinangle")
else:
    run("cd /content/spinangle && git pull", check=False)
os.chdir("/content/spinangle")

# 1) headless render libs ------------------------------------------------------
run("apt-get -qq update && apt-get -qq install -y xvfb zstd ffmpeg patchelf "
    "libegl1 libgl1-mesa-glx libosmesa6 libglfw3 libglew2.2 >/dev/null 2>&1", check=False)

# 2) uv + isolated Python 3.10 venv (reuse if present) -------------------------
if not os.path.exists(PY):
    run("pip install -q uv"); run("uv python install 3.10"); run(f"uv venv --python 3.10 {VENV}")

# 3) install [train] (required) + env deps (full [env] first, scoped fallback) --
if cap(f"uv pip install --python {PY} 'stable-worldmodel[train]' matplotlib huggingface_hub"):
    raise SystemExit("❌ [train] stack failed — paste the output above.")
if cap(f"uv pip install --python {PY} 'stable-worldmodel[env]'"):
    print(f"\n[note] full [env] won't resolve (bundles Crafter/craftax + Atari/ale-py, "
          f"not LeWM tasks). Installing this benchmark's env deps: {ENV_DEPS}")
    if cap(f"uv pip install --python {PY} {ENV_DEPS}"):
        raise SystemExit(f"❌ env deps for {BENCH} failed — paste the output above.")

# 4) verify imports; repair torch/torchvision ONLY if they're the problem ------
if cap(IMPORT_CHECK):
    print("\n[repair] reinstalling a consistent torch+torchvision pair (default CUDA wheels)...")
    run(f"uv pip install --python {PY} --reinstall-package torch --reinstall-package torchvision torch torchvision")
    if cap(IMPORT_CHECK):
        raise SystemExit("❌ imports still failing — paste the FAIL traceback(s) above for the exact fix.")

# 5) harness sanity ------------------------------------------------------------
run(f"{PY} smoke_test.py && {PY} metrics.py")

# 6) data + checkpoint; PHASE 1 reproduce official LeWM (eval renders -> xvfb) --
run(f"{PY} scripts/download_assets.py --benchmark {BENCH} --ckpt" + (" --data" if GET_DATA else ""))
run(f"xvfb-run -a {PY} eval.py --config-name={BENCH}.yaml policy={BENCH}/lewm")

# 7) train + eval variants -----------------------------------------------------
CKPT = f"{H}/checkpoints/{BENCH}"
for v in VARIANTS:
    run(f"{PY} train.py +experiment={v} data={DATACFG} "
        f"output_model_name={BENCH}/{v} trainer.max_epochs={EPOCHS} wandb.enabled=false")
    for old in sorted(glob.glob(f"{CKPT}/{v}/weights_epoch_*.pt"), key=os.path.getmtime)[:-1]:
        os.remove(old)
    run(f"xvfb-run -a {PY} eval.py --config-name={BENCH}.yaml policy={BENCH}/{v}")
    sph = "" if v in ("official_lewm", "lewm_nosigreg") else "--spherical"
    run(f"{PY} scripts/eval_latent_metrics.py --policy {BENCH}/{v} --data {DATACFG} "
        f"--benchmark {BENCH} --variant {v} {sph} --horizon 20 --num_batches 16", check=False)

# 8) plots ---------------------------------------------------------------------
run(f"{PY} scripts/make_plots.py")
from IPython.display import Image, display
for p in ["success_vs_steps", "rollout_error_vs_horizon", "retrieval_vs_steps",
          "rank_clumping", "planning_budget_curve"]:
    fp = f"/content/spinangle/plots/{p}.png"
    if os.path.exists(fp):
        display(Image(fp))
print("\n✅ DONE — results in results/all_runs.csv, plots in plots/.")


## Phase 7 — νGPT scaling (optional; run after the loop above)

In [ ]:
PY, BENCH, DATACFG, EPOCHS = '/content/lewmenv/bin/python', 'tworoom', 'tworoom', 100
for v in ['gated_spherical', 'ngpt_lr', 'ngpt_lr_groups']:
    !{PY} train.py +experiment={v} data={DATACFG} output_model_name={BENCH}/{v} \
        trainer.max_epochs={EPOCHS} wandb.enabled=false
    !xvfb-run -a {PY} eval.py --config-name={BENCH}.yaml policy={BENCH}/{v}


## Persist results back to the branch (optional)

In [ ]:
!cd /content/spinangle && git add results/all_runs.csv plots/*.png && \
  git -c user.email=colab@local -c user.name=colab commit -m 'colab: results' && \
  git push || echo 'configure git auth (token) to push'
